# Task 1: Article-Type Classification


## 1. Introduction

This notebook addresses **Task 1: Article-Type Classification** for Assignment 2 by predicting the article type of a fashion product from its RGB image, such as a shirt, dress, or pair of shoes. The task is formulated as **a multi-class image classification problem: each image receives one catalogue article-type label**.

Three neural-network approaches are developed and trained from scratch using TensorFlow/Keras:

- **Shallow MLP (Baseline):** A fully connected network with one 256-unit hidden layer operating on flattened image pixels.
- **Deeper MLP:** A fully connected network with hidden layers of 256, 128, and 64 units, using dropout to reduce overfitting.
- **Convolutional Neural Network (CNN):** A spatial model that learns local image patterns through convolutional layers. The CNN family includes three declared architecture/scheduling configurations.

Performance is assessed using:

- **Accuracy:** The proportion of correctly classified images.
- **Macro-F1:** The primary selection metric, giving equal weight to the F1 scores of classes present in the evaluation partition.
- **Per-class classification reports and confusion matrices:** Evidence of which labels are recognized reliably and which are confused.
- **Learning curves:** Training and validation loss, accuracy, and macro-F1 used to examine convergence and overfitting.
- **Calibration metrics:** Confidence reliability of the selected model, assessed after selection using separate validation groups.

Article categories can share silhouettes and textures, while rare categories have fewer training examples. The analysis therefore considers class support and common confusions alongside overall accuracy.

The workflow covers metadata inspection, preprocessing, model development, comparative evaluation, selection, and export for prediction. All candidates use the same frozen group-isolated data partitions. The internal test has prior development exposure, which remains a limitation after retraining. Numerical findings and the final model judgment must be completed from the new executed results; no winner is assumed in advance.


## 2. Library Imports & Setup

Use the Fashion Keras kernel. The seed controls initialization and augmentation. Float32 is used throughout; GPU availability is reported before models are created.


In [ ]:
# Locate the project, import the training libraries, select the accelerator, and fix random seeds.
from pathlib import Path
import sys, json
from concurrent.futures import ThreadPoolExecutor

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    log_loss,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import GroupShuffleSplit
from scipy.optimize import minimize_scalar
import tensorflow as tf
from tensorflow import keras
from scripts.preprocessing import (
    task_frame,
    IMAGE_SIZE,
    NORMALISATION_PATH,
    SEED,
    select_tensorflow_device,
)

DEVICE = select_tensorflow_device()
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
OUTPUT, RESULTS, FIGURES = (ROOT / 'models', ROOT / 'results', ROOT / 'figures')
for directory in (OUTPUT, RESULTS, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)
MAX_EPOCHS, BATCH_SIZE = (30, 64)

def stem_for(target):
    return 'article_type' if target == 'articleType' else target

# False for a fresh Run All. Enable only when deliberately resuming matching saved.
RESUME_SAVED_RESULTS = False

get_ipython().run_line_magic('matplotlib', 'inline')

## 3. Load Metadata

Load valid labelled images and their frozen split assignments. Inspect paths, target labels and missing values before preprocessing.


In [ ]:
# Load audited metadata and show how many labelled images are available in each frozen split.

metadata_by_target = {'articleType': task_frame('articleType')}
frame = metadata_by_target['articleType']
print('articleType', frame.shape)
display(frame.head())


## 4. Data Preprocessing

Prepare the same images and label encoding for every candidate. Preserve group isolation and fit normalization on training data only.


### 4.1. Class Distribution & Balancing Strategy

Preserve the original sample distribution. Initial candidates use unweighted cross-entropy; additional CNN experiments compare this with capped square-root inverse-frequency weights computed only from training counts. No examples are duplicated, and validation metrics remain unweighted.


In [ ]:
# Measure training-set class imbalance before choosing architectures and evaluation metrics.
train_rows = metadata_by_target['articleType'].loc[
    metadata_by_target['articleType']['split'].eq('train')
]
counts = train_rows['articleType'].value_counts()
counts.head(25).sort_values().plot.barh(figsize=(8, 6), title=f"{'articleType'}: training support")
plt.tight_layout()
plt.show()

### 4.2. Training, Validation & Test Partitions

Reuse the frozen product-group split. Within validation, use separate groups for selection, temperature fitting and policy checking. Do not use the internal test to select candidates.


In [ ]:
# Divide validation groups into model-selection and calibration views without splitting related products.
def validation_views(frame):
    selection_ids, rest_ids = next(
        GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED).split(
            frame, groups=frame.group_key
        )
    )
    selection, rest = frame.iloc[selection_ids], frame.iloc[rest_ids]
    cal_ids, policy_ids = next(
        GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED + 1).split(
            rest, groups=rest.group_key
        )
    )
    return selection, rest.iloc[cal_ids], rest.iloc[policy_ids]

In [ ]:
# Create the split views, fixed label order, and reusable batch loaders for each target.
frames, labels_by_target = ({}, {})
selection, calibration, policy = validation_views(
    metadata_by_target['articleType'].loc[
        metadata_by_target['articleType']['split'].eq('validation')
    ]
)
frames['articleType'] = dict(
    train=metadata_by_target['articleType'].loc[
        metadata_by_target['articleType']['split'].eq('train')
    ],
    selection=selection,
    calibration=calibration,
    policy=policy,
)
groups = [set(frame.group_key) for frame in frames['articleType'].values()]
assert all((a.isdisjoint(b) for i, a in enumerate(groups) for b in groups[i + 1 :]))
labels_by_target['articleType'] = sorted(frames['articleType']['train']['articleType'].unique())
display(
    pd.Series(
        {name: len(frame) for name, frame in frames['articleType'].items()}, name='articleType'
    )
)

### 4.3. Image Preprocessing

Resize RGB to 96 x 128 pixels (width x height), then normalize using training-only channel statistics. The batch class stores decoded uint8 images and converts one batch at a time to float32. Horizontal flips are applied inside each model only during training.


In [ ]:
# Cache resized images in RAM and serve normalized batches to Keras during training and evaluation.
class CachedBatches(keras.utils.PyDataset):
    """Decode RGB once to uint8 RAM, then normalize each NHWC batch."""

    def __init__(self, frame, target, labels, normalisation, training=False, batch_size=64):
        super().__init__(workers=0, max_queue_size=2)
        self.dataset = frame
        self.training, self.batch_size = training, batch_size

        def read(path):
            with Image.open(path) as image:
                return np.asarray(
                    image.convert("RGB").resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
                ).copy()

        # Decode and resize each source image once; later epochs reuse this cache.
        with ThreadPoolExecutor(max_workers=4) as pool:
            self.images = np.stack(list(pool.map(read, frame.image_path)))

        # Convert string labels to the fixed integer order expected by the output layer.
        self.targets = np.asarray([labels.index(label) for label in frame[target]], dtype=np.int32)
        self.mean = np.asarray(normalisation["mean"], dtype=np.float32)
        self.std = np.asarray(normalisation["std"], dtype=np.float32)
        self.reset()

    def reset(self):
        self.rng = np.random.default_rng(SEED)
        self.indices = np.arange(len(self.dataset))
        if self.training:
            self.rng.shuffle(self.indices)

    def __len__(self):
        return (len(self.dataset) + self.batch_size - 1) // self.batch_size

    # Normalize only the requested batch to avoid storing a second float32 image copy.
    def __getitem__(self, index):
        ids = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        images = self.images[ids].astype(np.float32) / 255.0
        return (images - self.mean) / self.std, self.targets[ids]

    def on_epoch_end(self):
        if self.training:
            self.rng.shuffle(self.indices)

### 4.4. Feature Batches & Label Encoding

Map sorted label names to integer indices, preserving the same order for every model. Construct shuffled training batches and fixed-order selection batches.


In [ ]:
# Load normalization statistics computed from training images only, then construct all data loaders.
normalisation = json.loads(NORMALISATION_PATH.read_text())
loaders = {}
labels = labels_by_target['articleType']
loaders['articleType'] = {
    name: CachedBatches(
        frames['articleType'][name],
        'articleType',
        labels,
        normalisation,
        training=name == 'train',
        batch_size=BATCH_SIZE,
    )
    for name in ['train', 'selection']
}

## 5. Model Development & Evaluation

Train and evaluate a shallow MLP baseline, a deeper MLP and a CNN using the same image preprocessing and frozen partitions. Architecture construction, fitting, class-level evaluation, calibration and model export are implemented in this notebook.


### Evaluation Metric: Macro-F1

Accumulate the full-epoch confusion matrix and average F1 over classes with positive ground-truth support. This metric selects the restored epoch and winning model. It does not average per-batch F1 scores.

The displayed per-class classification report also includes every output label for coverage auditing. Its standard `macro avg` row therefore includes zero-support output labels and can differ from the supported-class macro-F1 used for selection. Read the explicitly reported selection metric for model ranking, and inspect support before making claims about all classes. Selection support is 97/124 article types, 4/4 seasons, 5/5 gender categories and 8/9 occasion labels for this frozen run.


In [ ]:
# Track macro-F1 across a complete epoch with a running confusion matrix.
@keras.utils.register_keras_serializable(package="Fashion")
class SupportedMacroF1(keras.metrics.Metric):
    """Macro-F1 over classes present in the full epoch's ground truth."""

    def __init__(self, num_classes, name="macro_f1", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.matrix = self.add_weight(
            name="matrix", shape=(num_classes, num_classes), initializer="zeros"
        )

    # Accumulate one confusion matrix across all batches in the epoch.
    def update_state(self, y_true, y_pred, sample_weight=None):
        truth = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        predictions = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        weights = (
            None if sample_weight is None else tf.cast(tf.reshape(sample_weight, [-1]), self.dtype)
        )
        self.matrix.assign_add(
            tf.math.confusion_matrix(
                truth, predictions, self.num_classes, weights=weights, dtype=self.dtype
            )
        )

    # Average F1 only across classes that are present in the ground truth.
    def result(self):
        support = tf.reduce_sum(self.matrix, axis=1)
        predicted = tf.reduce_sum(self.matrix, axis=0)
        f1 = tf.math.divide_no_nan(2 * tf.linalg.diag_part(self.matrix), support + predicted)
        mask = tf.cast(support > 0, self.dtype)
        return tf.math.divide_no_nan(tf.reduce_sum(f1 * mask), tf.reduce_sum(mask))

    def reset_state(self):
        self.matrix.assign(tf.zeros_like(self.matrix))

    def get_config(self):
        return {**super().get_config(), "num_classes": self.num_classes}

### Saved-Model Metadata

An identity layer stores label order, preprocessing and confidence policy inside each exported Keras model.


In [ ]:
# Attach labels and preprocessing settings to the exported Keras model as a pass-through layer.
@keras.utils.register_keras_serializable(package="Fashion")
class ModelMetadata(keras.layers.Layer):
    """Store label order, preprocessing and calibration inside the .keras file."""

    def __init__(self, metadata=None, **kwargs):
        super().__init__(**kwargs)
        self.metadata = dict(metadata or {})

    def call(self, inputs):
        return inputs

    def get_config(self):
        return {**super().get_config(), "metadata": dict(self.metadata)}

In [ ]:
# Keep trained models, learning histories, and selection scores organized by target and method.
models = {'articleType': {}}
histories = {'articleType': {}}
selection_scores = {}

### Configuration rationale and evaluation protocol

Each target compares three neural model families. Dense width and depth control model capacity; convolutional blocks preserve spatial relationships. Configuration choices follow validation macro-F1, then accuracy, then parameter count. The selected architectures and training settings are specified below; they are empirical choices, not theoretically optimal designs.

The 96 ? 128 RGB input and training-only normalization are shared across candidates. Most source images are only 60 ? 80, so enlarging them further would not recover additional image detail. Output width is determined by the target's training label vocabulary, rather than tuned independently.

Where required, CNN training has an initial four-block stage followed by a lower-learning-rate continuation. Both stages are implemented here. Class weights use training counts only. The initial stage is part of the training recipe, not a fourth model family.

The recorded tables summarize completed runs of these configurations. This reorganized notebook has not yet been executed end-to-end; rerunning the cells regenerates its metrics, figures and exports. Single-seed results may vary. Only the supplied dataset, shared preprocessing module and frozen Task 0 preprocessing artifacts are required; all model training code is included here.


### General design justification

**Input and output.** A fixed 96 ? 128 RGB input standardizes batching and inference while keeping computation manageable. Its portrait aspect ratio matches the dominant 60 ? 80 source format. Resizing cannot recover missing detail, so larger inputs are not assumed to improve recognition. Output units match the target's label vocabulary; they are not a freely tuned capacity setting.

**Dense baselines.** Flattening provides a straightforward pixel-based baseline. A shallow MLP tests whether one nonlinear hidden layer is sufficient; a deeper MLP tests additional nonlinear transformations. Dense layers do not explicitly preserve local spatial relationships, and flattening creates large parameter counts. ReLU introduces nonlinearity. Hidden widths and depths are selected using validation evidence, not the number of classes alone.

**CNN architecture.** Small 3 ? 3 convolutions share weights across positions and can learn local appearance patterns. Successive blocks allow information from larger image regions to be combined. Max pooling reduces spatial dimensions and computational cost, but can discard fine detail. The final 2 ? 2 average-pooled grid limits the dense head's size while retaining a coarse spatial layout. Batch normalization stabilizes intermediate activation scales. These design properties motivate CNN use; they do not prove that a particular learned filter detects a specific garment feature.

**Regularization and optimization.** Dropout 0.2 discourages reliance on individual hidden activations, while training-only horizontal flips expose models to mirrored product views. These are attempts to improve generalization, not guarantees. Adam adapts parameter updates; reducing its learning rate on a validation plateau allows smaller updates later in CNN training. Early stopping restores the best selection macro-F1 epoch and limits training after improvement stalls. The fixed seed supports repeatability, and batch size 64 is a practical training setting rather than a demonstrated optimum.

**Model selection.** Group-isolated partitions reduce related-product leakage, training-only normalization avoids fitting preprocessing to evaluation data, and macro-F1 gives each supported class equal weight. Accuracy, class-level errors and parameter counts provide complementary evidence. Different labels may favour different capacities; the measured validation comparison justifies the exact choices, while the general principles above explain their intended roles.


In [ ]:
SELECTED_CONFIGS = {
    'articleType': {
        'shallow_mlp': {
            'method': 'shallow_mlp_lower_lr',
            'widths': [256],
            'learning_rate': 0.0001,
            'epochs': 20,
            'patience': 4,
        },
        'deeper_mlp': {
            'method': 'deeper_mlp_two_layers',
            'widths': [256, 128],
            'learning_rate': 0.0003,
            'epochs': 20,
            'patience': 4,
        },
        'cnn': {
            'method': 'cnn_continue_weighted',
            'widths': [256],
            'learning_rate': 3e-05,
            'epochs': 30,
            'patience': 7,
        },
    }
}

### 5.1. Shallow MLP (Baseline)

Flattened RGB input is passed through the selected dense layers, with ReLU and dropout 0.2 after each hidden layer.


#### 5.1.1. Article Type: Model Architecture & Training Configuration

Configuration `shallow_mlp_lower_lr`; hidden dense units [256]. Adam 0.0001, maximum 20 epochs, early-stopping patience 4.


In [ ]:
# Build the shallow MLP from the selected target-specific configuration.
config = SELECTED_CONFIGS['articleType']['shallow_mlp']
method = config['method']
keras.utils.set_random_seed(SEED)

# Start with a fixed image input and training-only horizontal augmentation.
layers = [
    keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)),
    keras.layers.RandomFlip('horizontal'),
]
layers.append(keras.layers.Flatten())

# Add the configured hidden classifier layers and dropout regularization.
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend(
    [keras.layers.Dense(len(labels_by_target['articleType'])), ModelMetadata(name='metadata')]
)
models['articleType'][method] = keras.Sequential(layers, name=method)
models['articleType'][method].summary()

#### 5.1.2. Compile & Train Model

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
# Compile and train the model while restoring the epoch with the best selection macro-F1.
model = models['articleType'][method]

# Configure optimization and report both accuracy and imbalance-aware macro-F1.
model.compile(
    optimizer=keras.optimizers.Adam(0.0001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        SupportedMacroF1(len(labels_by_target['articleType'])),
    ],
)
loaders['articleType']['train'].reset()

# Stop when selection macro-F1 no longer improves and restore its best weights.
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', mode='max', patience=4, restore_best_weights=True
    )
]

# Keep the selection partition read-only: it chooses epochs but never updates weights.
fitted = model.fit(
    loaders['articleType']['train'],
    validation_data=loaders['articleType']['selection'],
    epochs=20,
    callbacks=callbacks,
    shuffle=False,
    verbose=2,
)
histories['articleType'][method] = pd.DataFrame(fitted.history)

#### 5.1.3. Evaluation and Performance Metrics

Calculate validation accuracy and supported-class macro-F1, display the per-class classification report, and plot training and validation loss, accuracy and macro-F1. These measurements use the restored best epoch.


In [ ]:
# Evaluate the restored best weights, show class metrics, and plot the learning curves.
loader = loaders['articleType']['selection']

# Convert logits to probabilities and preserve the loader's row order.
probabilities = np.vstack(
    [
        tf.nn.softmax(models['articleType'][method](loader[i][0], training=False)).numpy()
        for i in range(len(loader))
    ]
)
predictions = probabilities.argmax(1)
display(
    pd.Series(
        dict(
            selection_accuracy=accuracy_score(loader.targets, predictions),
            selection_macro_f1=f1_score(
                loader.targets,
                predictions,
                labels=np.unique(loader.targets),
                average='macro',
                zero_division=0,
            ),
        )
    )
)

# Inspect every label because aggregate accuracy can hide minority-class failures.
per_class = pd.DataFrame(
    classification_report(
        loader.targets,
        predictions,
        labels=np.arange(len(labels_by_target['articleType'])),
        target_names=labels_by_target['articleType'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('articleType')}_{method}_selection_per_class.csv")

# Plot train and selection curves to diagnose convergence and overfitting.
history = histories['articleType'][method]
history.to_csv(RESULTS / f"{stem_for('articleType')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history) + 1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history) + 1), history['val_' + metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('articleType')}_{method}_learning_curves.png", dpi=160)
plt.show()

# Store one comparable row for the final cross-family decision.
selection_scores.setdefault('articleType', {})[method] = dict(
    family='shallow_mlp',
    validation_accuracy=accuracy_score(loader.targets, predictions),
    validation_macro_f1=f1_score(
        loader.targets,
        predictions,
        labels=np.unique(loader.targets),
        average='macro',
        zero_division=0,
    ),
    complexity_parameters=models['articleType'][method].count_params(),
    epochs_run=len(history),
)

#### 5.1.4. Evaluation Analysis

**Learning behaviour.** The shallow MLP reached its best selection macro-F1 at epoch 11 of 15. At that point, training accuracy was **75.82%**, selection accuracy was **74.21%**, and the corresponding losses were **0.8191** and **0.9979**. The small accuracy gap suggests that dropout and augmentation controlled severe overfitting, although the higher selection loss shows that some incorrect predictions remained highly confident. The model achieved **0.4432 macro-F1**, substantially below its accuracy, so its performance was not distributed evenly across the article-type classes.

**Class-level behaviour.** The model classified visually distinctive, well-represented products reliably. Watches achieved precision/recall/F1 of **0.963/0.972/0.967** (213 images), while Bra and Sunglasses both obtained F1 above **0.96**. Performance was much weaker for less frequent or visually overlapping categories: Bangle recall was **0.077**, Dresses recall was **0.107**, and Capris recall was **0.143**. Among the 97 classes represented in the selection partition, 28 received zero recall. This explains why the accuracy remains respectable while macro-F1 is much lower.

**Interpretation.** Flattening the image allows this model to form a useful baseline, but it discards the explicit neighbourhood structure between pixels. It therefore learns dominant global patterns and frequent categories more readily than subtle shape differences. The result is suitable as an ANN baseline, but its limited rare-class coverage makes it a weak candidate for the deployed article-type classifier.


### 5.2. Deeper MLP

Flattened RGB input is passed through the selected dense layers, with ReLU and dropout 0.2 after each hidden layer.


#### 5.2.1. Article Type: Model Architecture & Training Configuration

Configuration `deeper_mlp_two_layers`; hidden dense units [256, 128]. Adam 0.0003, maximum 20 epochs, early-stopping patience 4.


In [ ]:
# Build the deeper MLP from the selected target-specific configuration.
config = SELECTED_CONFIGS['articleType']['deeper_mlp']
method = config['method']
keras.utils.set_random_seed(SEED)

# Start with a fixed image input and training-only horizontal augmentation.
layers = [
    keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)),
    keras.layers.RandomFlip('horizontal'),
]
layers.append(keras.layers.Flatten())

# Add the configured hidden classifier layers and dropout regularization.
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend(
    [keras.layers.Dense(len(labels_by_target['articleType'])), ModelMetadata(name='metadata')]
)
models['articleType'][method] = keras.Sequential(layers, name=method)
models['articleType'][method].summary()

#### 5.2.2. Compile & Train Model

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
# Compile and train the model while restoring the epoch with the best selection macro-F1.
model = models['articleType'][method]

# Configure optimization and report both accuracy and imbalance-aware macro-F1.
model.compile(
    optimizer=keras.optimizers.Adam(0.0003),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        SupportedMacroF1(len(labels_by_target['articleType'])),
    ],
)
loaders['articleType']['train'].reset()

# Stop when selection macro-F1 no longer improves and restore its best weights.
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', mode='max', patience=4, restore_best_weights=True
    )
]

# Keep the selection partition read-only: it chooses epochs but never updates weights.
fitted = model.fit(
    loaders['articleType']['train'],
    validation_data=loaders['articleType']['selection'],
    epochs=20,
    callbacks=callbacks,
    shuffle=False,
    verbose=2,
)
histories['articleType'][method] = pd.DataFrame(fitted.history)

#### 5.2.3. Evaluation and Performance Metrics

Calculate validation accuracy and supported-class macro-F1, display the per-class classification report, and plot training and validation loss, accuracy and macro-F1. These measurements use the restored best epoch.


In [ ]:
# Evaluate the restored best weights, show class metrics, and plot the learning curves.
loader = loaders['articleType']['selection']

# Convert logits to probabilities and preserve the loader's row order.
probabilities = np.vstack(
    [
        tf.nn.softmax(models['articleType'][method](loader[i][0], training=False)).numpy()
        for i in range(len(loader))
    ]
)
predictions = probabilities.argmax(1)
display(
    pd.Series(
        dict(
            selection_accuracy=accuracy_score(loader.targets, predictions),
            selection_macro_f1=f1_score(
                loader.targets,
                predictions,
                labels=np.unique(loader.targets),
                average='macro',
                zero_division=0,
            ),
        )
    )
)

# Inspect every label because aggregate accuracy can hide minority-class failures.
per_class = pd.DataFrame(
    classification_report(
        loader.targets,
        predictions,
        labels=np.arange(len(labels_by_target['articleType'])),
        target_names=labels_by_target['articleType'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('articleType')}_{method}_selection_per_class.csv")

# Plot train and selection curves to diagnose convergence and overfitting.
history = histories['articleType'][method]
history.to_csv(RESULTS / f"{stem_for('articleType')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history) + 1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history) + 1), history['val_' + metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('articleType')}_{method}_learning_curves.png", dpi=160)
plt.show()

# Store one comparable row for the final cross-family decision.
selection_scores.setdefault('articleType', {})[method] = dict(
    family='deeper_mlp',
    validation_accuracy=accuracy_score(loader.targets, predictions),
    validation_macro_f1=f1_score(
        loader.targets,
        predictions,
        labels=np.unique(loader.targets),
        average='macro',
        zero_division=0,
    ),
    complexity_parameters=models['articleType'][method].count_params(),
    epochs_run=len(history),
)

#### 5.2.4. Evaluation Analysis

**Learning behaviour.** The deeper MLP recorded its best macro-F1 at epoch 20, the final scheduled epoch. Training and selection accuracy were **72.45%** and **73.24%**, with losses of **0.9155** and **0.9279**. These closely aligned curves provide little evidence of overfitting; validation performance slightly exceeding training performance is plausible because dropout and augmentation make the training batches harder. However, the final selection macro-F1 was only **0.3783**, below the shallow MLP despite the extra hidden layer.

**Class-level behaviour.** Strong results again concentrated in distinctive categories: Sunglasses and Bra achieved F1 scores of **0.984** and **0.983**, and Watches achieved **0.959**. In contrast, Capris, Flats and Sweaters achieved F1 scores of only **0.100**, **0.105** and **0.118**. The model produced zero recall for 38 of the 97 represented classes, ten more than the shallow MLP. Its final-epoch peak suggests that further optimization might still move the training objective, but the broad failure on minority categories indicates that additional dense depth alone is not solving the central representation problem.

**Interpretation.** The deeper network has more nonlinear capacity, yet it still receives a flattened image and must learn spatial relationships indirectly. Its lower accuracy and macro-F1 show that extra dense layers did not translate into better generalization for this target. Consequently, the deeper MLP is informative as a DNN comparison but is not preferred over either the shallow baseline or the CNN.


### 5.3. Convolutional Neural Network (CNN)

The selected CNN uses four 3 ? 3 convolution blocks (32, 64, 128, 256 filters), batch normalization, ReLU and max pooling, followed by a 2 ? 2 pooled grid and Dense(256). Only the selected continuation, when required, is fitted.


#### 5.3.1. Article Type: Model Architecture & Training Configuration

Configuration `cnn_continue_weighted`; hidden dense units [256]. Initial Adam 0.001, maximum 30 epochs, early-stopping patience 7 and learning-rate halving after two unimproved epochs. Continuation, if specified, uses eight additional epochs with patience 4 and a fresh optimizer.


In [ ]:
# Build the CNN from the selected target-specific configuration.
config = SELECTED_CONFIGS['articleType']['cnn']
method = config['method']
keras.utils.set_random_seed(SEED)

# Start with a fixed image input and training-only horizontal augmentation.
layers = [
    keras.Input(shape=(IMAGE_SIZE[1], IMAGE_SIZE[0], 3)),
    keras.layers.RandomFlip('horizontal'),
]

# Learn increasingly abstract local features while reducing spatial resolution.
for width in [32, 64, 128, 256]:
    layers.extend(
        [
            keras.layers.Conv2D(width, 3, padding='same'),
            keras.layers.BatchNormalization(),
            keras.layers.Activation('relu'),
            keras.layers.MaxPooling2D(2),
        ]
    )
h, w = IMAGE_SIZE[1] // 16, IMAGE_SIZE[0] // 16
layers.extend([keras.layers.AveragePooling2D((h // 2, w // 2)), keras.layers.Flatten()])

# Add the configured hidden classifier layers and dropout regularization.
for width in config['widths']:
    layers.extend([keras.layers.Dense(width, activation='relu'), keras.layers.Dropout(0.2)])
layers.extend(
    [keras.layers.Dense(len(labels_by_target['articleType'])), ModelMetadata(name='metadata')]
)
models['articleType'][method] = keras.Sequential(layers, name=method)
models['articleType'][method].summary()

#### 5.3.2. Compile & Train Model

The selection view controls epoch selection. Test images are excluded from training.


In [ ]:
# Compile and train the model while restoring the epoch with the best selection macro-F1.
model = models['articleType'][method]

# Configure optimization and report both accuracy and imbalance-aware macro-F1.
model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        SupportedMacroF1(len(labels_by_target['articleType'])),
    ],
)
loaders['articleType']['train'].reset()

# Stop when selection macro-F1 no longer improves and restore its best weights.
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', mode='max', patience=7, restore_best_weights=True
    )
]
callbacks.insert(
    0, keras.callbacks.ReduceLROnPlateau(monitor='val_macro_f1', mode='max', factor=0.5, patience=2)
)

# Keep the selection partition read-only: it chooses epochs but never updates weights.
fitted = model.fit(
    loaders['articleType']['train'],
    validation_data=loaders['articleType']['selection'],
    epochs=30,
    callbacks=callbacks,
    shuffle=False,
    verbose=2,
)
histories['articleType'][method] = pd.DataFrame(fitted.history)
histories['articleType'][method].to_csv(
    RESULTS / 'article_type_cnn_initial_history.csv', index=False
)

# Begin the low-learning-rate continuation from the initial CNN weights.
# Reset the loader so continuation starts from a reproducible order.
keras.utils.set_random_seed(SEED)
loaders['articleType']['train'].reset()
model.compile(
    optimizer=keras.optimizers.Adam(3e-05),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        SupportedMacroF1(len(labels_by_target['articleType'])),
    ],
)
class_weight = None

# Use capped square-root class weights to help rare labels without destabilizing training.
counts = np.bincount(
    loaders['articleType']['train'].targets, minlength=len(labels_by_target['articleType'])
)
weights = np.sqrt(counts.sum() / (len(counts) * counts))
weights = np.clip(weights / np.average(weights, weights=counts), 0.25, 4)
class_weight = dict(enumerate(weights))
fitted = model.fit(
    loaders['articleType']['train'],
    validation_data=loaders['articleType']['selection'],
    epochs=8,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_macro_f1', mode='max', patience=4, restore_best_weights=True
        )
    ],
    class_weight=class_weight,
    shuffle=False,
    verbose=2,
)
histories['articleType'][method] = pd.DataFrame(fitted.history)

#### 5.3.3. Evaluation and Performance Metrics

Calculate validation accuracy and supported-class macro-F1, display the per-class classification report, and plot training and validation loss, accuracy and macro-F1. These measurements use the restored best epoch.


In [ ]:
# Evaluate the restored best weights, show class metrics, and plot the learning curves.
loader = loaders['articleType']['selection']

# Convert logits to probabilities and preserve the loader's row order.
probabilities = np.vstack(
    [
        tf.nn.softmax(models['articleType'][method](loader[i][0], training=False)).numpy()
        for i in range(len(loader))
    ]
)
predictions = probabilities.argmax(1)
display(
    pd.Series(
        dict(
            selection_accuracy=accuracy_score(loader.targets, predictions),
            selection_macro_f1=f1_score(
                loader.targets,
                predictions,
                labels=np.unique(loader.targets),
                average='macro',
                zero_division=0,
            ),
        )
    )
)

# Inspect every label because aggregate accuracy can hide minority-class failures.
per_class = pd.DataFrame(
    classification_report(
        loader.targets,
        predictions,
        labels=np.arange(len(labels_by_target['articleType'])),
        target_names=labels_by_target['articleType'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / f"{stem_for('articleType')}_{method}_selection_per_class.csv")

# Plot train and selection curves to diagnose convergence and overfitting.
history = histories['articleType'][method]
history.to_csv(RESULTS / f"{stem_for('articleType')}_{method}_history.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
    ax.plot(np.arange(1, len(history) + 1), history[metric], label='Train')
    ax.plot(np.arange(1, len(history) + 1), history['val_' + metric], label='Selection')
    ax.set(xlabel='Epoch', ylabel=metric, title=method)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / f"{stem_for('articleType')}_{method}_learning_curves.png", dpi=160)
plt.show()

# Store one comparable row for the final cross-family decision.
selection_scores.setdefault('articleType', {})[method] = dict(
    family='cnn',
    validation_accuracy=accuracy_score(loader.targets, predictions),
    validation_macro_f1=f1_score(
        loader.targets,
        predictions,
        labels=np.unique(loader.targets),
        average='macro',
        zero_division=0,
    ),
    complexity_parameters=models['articleType'][method].count_params(),
    epochs_run=len(history),
)

#### 5.3.4. Evaluation Analysis

**Learning behaviour.** During the recorded continuation stage, the CNN reached its best selection macro-F1 at epoch 6 of 8. Training accuracy was **96.34%** and selection accuracy was **87.00%**; training and selection losses were **0.1330** and **0.5412**. The roughly nine-point accuracy gap indicates some overfitting, so early stopping and retention of the best weights are important. Even with this gap, the model achieved **0.6954 macro-F1**, a large improvement over both MLPs and evidence that its convolutional features generalize more effectively.

**Class-level behaviour.** The CNN was strong on both frequent and several moderately supported categories. Bra and Accessory Gift Set achieved perfect precision and recall on 29 and 12 selection images, while Sunglasses achieved **0.994 F1**. For high-support classes, Tshirts reached **0.950 precision**, **0.930 recall** and **0.940 F1**, and Shirts reached **0.928 F1**. Remaining weaknesses were concentrated in difficult minority classes: Bangle recall was **0.154**, Capris recall was **0.357**, and Jackets recall was **0.455**. Thirteen represented classes had zero recall, which is still a limitation but is substantially fewer than the shallow MLP's 28 and the deeper MLP's 38.

**Interpretation.** Convolution preserves local shape, edge and texture relationships before classification, which is better matched to distinguishing product types than flattening raw pixels. Class weighting also improves attention to minority categories, though extremely small classes remain unstable. Its clear lead in both accuracy and macro-F1 makes this CNN the strongest article-type candidate despite the observed train–selection gap.


## 6. Ultimate Judgement

The three model families are compared using the validation metrics already calculated in their evaluation sections. The model with the highest validation macro-F1 is selected. Accuracy and parameter count are used to interpret the result and would break an exact macro-F1 tie.


In [ ]:
# Prepare containers for the cross-family comparison and final checkpoints.
comparisons, winners, checkpoints = {}, {}, {}

In [ ]:
# Rank the three model families by macro-F1, then accuracy, then parameter count.
comparison = pd.DataFrame(selection_scores['articleType']).T
comparison = comparison.sort_values(
    ['validation_macro_f1', 'validation_accuracy', 'complexity_parameters'],
    ascending=[False, False, True],
)
winners['articleType'] = comparison.index[0]
comparison['selected'] = comparison.index == winners['articleType']
comparisons['articleType'] = comparison
display(comparison)
print('Selected:', winners['articleType'])
comparison.to_csv(RESULTS / 'article_type_comparison.csv')

**Model comparison: articleType**

| Family | Configuration | Validation accuracy | Macro-F1 | Parameters |
|---|---|---:|---:|---:|
| shallow_mlp | shallow_mlp_lower_lr | 74.21% | 0.4432 | 9,469,308 |
| deeper_mlp | deeper_mlp_two_layers | 73.24% | 0.3783 | 9,486,332 |
| cnn | cnn_continue_weighted | 87.00% | 0.6954 | 684,604 |


In [ ]:
# Define imbalance-aware performance and probability-calibration metrics for the final test.
def supported_macro_f1(truth, predictions) -> float:
    """Macro-average over labels present in the ground-truth partition."""
    return float(
        f1_score(truth, predictions, labels=np.unique(truth), average="macro", zero_division=0)
    )


def expected_calibration_error(
    truth: np.ndarray,
    probabilities: np.ndarray,
    bins: int = 10,
) -> float:
    """Compute top-label expected calibration error."""
    truth = np.asarray(truth)
    probabilities = np.asarray(probabilities)
    confidence = probabilities.max(axis=1)
    correct = probabilities.argmax(axis=1) == truth
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = len(truth)
    error = 0.0
    for lower, upper in zip(edges[:-1], edges[1:], strict=True):
        selected = (confidence > lower) & (confidence <= upper)
        if selected.any():
            error += (
                selected.sum()
                / total
                * abs(float(correct[selected].mean()) - float(confidence[selected].mean()))
            )
    return float(error)

### 6.1. Selected-Model Evaluation

Evaluate only the selected model on the internal test partition. This cell reports accuracy, supported-class macro-F1, calibration error, negative log-likelihood, Brier score and a per-class classification report. The internal test has prior development exposure and is not used to revise the selected architecture.


In [ ]:
# Evaluate each selected family winner once on the untouched internal-test partition.
test_results = {}
checkpoints = {}

method = winners['articleType']
model = models['articleType'][method]
test_frame = metadata_by_target['articleType'].loc[
    metadata_by_target['articleType']['split'].eq('test')
]
test_loader = CachedBatches(
    test_frame, 'articleType', labels_by_target['articleType'], normalisation, batch_size=BATCH_SIZE
)

# Convert logits to probabilities and preserve the loader's row order.
probabilities = np.vstack(
    [
        tf.nn.softmax(model(test_loader[i][0], training=False)).numpy()
        for i in range(len(test_loader))
    ]
)
truth = test_loader.targets
predictions = probabilities.argmax(1)

# Measure label accuracy, class balance, and probability quality on the final test.
metrics = {
    'accuracy': float(accuracy_score(truth, predictions)),
    'macro_f1': supported_macro_f1(truth, predictions),
    'ece': expected_calibration_error(truth, probabilities),
    'nll': float(
        log_loss(truth, probabilities, labels=np.arange(len(labels_by_target['articleType'])))
    ),
    'brier': float(
        np.mean(
            np.sum(
                (probabilities - np.eye(len(labels_by_target['articleType']))[truth]) ** 2, axis=1
            )
        )
    ),
}
test_results['articleType'] = dict(
    labels=labels_by_target['articleType'],
    truth=truth,
    predictions=predictions,
    probabilities=probabilities,
    metrics=metrics,
)

# Bundle the fitted model with everything required to reproduce inference.
checkpoints['articleType'] = dict(
    target='articleType',
    labels=labels_by_target['articleType'],
    model=model,
    model_type=method,
    temperature=1.0,
    review_policy=None,
    mean=normalisation['mean'],
    std=normalisation['std'],
    image_size=list(IMAGE_SIZE),
    comparison_row=comparisons['articleType'].loc[method].to_dict(),
    test_metrics=metrics,
)
display(pd.Series(metrics, name='articleType'))

# Inspect every label because aggregate accuracy can hide minority-class failures.
per_class = pd.DataFrame(
    classification_report(
        truth,
        predictions,
        labels=np.arange(len(labels_by_target['articleType'])),
        target_names=labels_by_target['articleType'],
        output_dict=True,
        zero_division=0,
    )
).T
display(per_class)
per_class.to_csv(RESULTS / 'article_type_test_per_class.csv')
pd.Series(metrics).to_csv(RESULTS / 'article_type_test_metrics.csv')

### 6.2. Selected-Model Performance

**Recorded selected-model performance.**

| Target | Selected model | Validation accuracy | Validation macro-F1 | Internal-test accuracy | Internal-test macro-F1 |
|---|---|---:|---:|---:|---:|
| articleType | cnn_continue_weighted | 87.00% | 0.6954 | 86.34% | 0.6373 |

Models are selected by validation macro-F1. Internal-test scores have prior development exposure and are descriptive, not an independent model-selection criterion.


In [ ]:
# Display the concise final judgement for the chosen model and its internal-test result.
winner = comparisons['articleType'].loc[winners['articleType']]
baseline = (
    comparisons['articleType'].loc[comparisons['articleType']['family'].eq('shallow_mlp')].iloc[0]
)
display(
    pd.Series(
        {
            'selected_method': winners['articleType'],
            'selection_macro_f1': winner.validation_macro_f1,
            'selection_accuracy': winner.validation_accuracy,
            'macro_f1_gain_over_baseline': winner.validation_macro_f1
            - baseline.validation_macro_f1,
            'internal_test_accuracy': test_results['articleType']['metrics']['accuracy'],
            'internal_test_macro_f1': test_results['articleType']['metrics']['macro_f1'],
        },
        name='articleType',
    )
)

### 6.3. Decision Analysis

The article-type CNN achieves 0.6954 validation macro-F1, compared with 0.4432 for the 256-unit shallow MLP and 0.3783 for the two-layer deeper MLP (256, 128). Its 684,604 parameters are substantially fewer than the roughly 9.5 million in either dense model. Spatial processing is a useful design choice for category recognition, although these metrics alone do not reveal the features learned by individual filters.

The two-layer deeper MLP is retained because it improved validation macro-F1 over the tested three-layer lower-rate configuration (0.3160). The 256-unit shallow baseline remained stronger than its smaller alternatives. Weighted CNN continuation balances the declared minority-sensitive selection objective against overall accuracy. These observations justify the chosen configurations empirically; additional depth or filter width is not automatically beneficial.

The selected CNN records 86.34% internal-test accuracy and 0.6373 macro-F1. The gap between accuracy and macro-F1 limits claims about rare article categories. The internal test has prior development exposure, and retrieval quality requires its own evaluation.


## 7. Final Prediction

Save the selected Keras model with its label order and preprocessing metadata, reload it, and run one sample prediction. This functional check uses softmax probabilities with temperature 1.0; it does not add a separate confidence-calibration or review policy.


In [ ]:
# Export a self-contained inference model with preprocessing and label metadata embedded.
def save_checkpoint(checkpoint, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path = path.with_suffix(".keras")
    model = checkpoint["model"]

    # Separate serializable metadata from the live Keras model object.
    metadata = {key: value for key, value in checkpoint.items() if key != "model"}
    metadata = json.loads(json.dumps(metadata, default=lambda value: value.item()))
    model.get_layer("metadata").metadata = metadata

    # Export architecture, learned weights and metadata without optimizer slots.
    # Rebuild without optimizer state to keep the deployment file smaller.
    inference_model = type(model).from_config(model.get_config())
    inference_model.set_weights(model.get_weights())
    inference_model.save(path)
    return path

In [ ]:
# Save the selected model, its learning history, and a machine-readable experiment summary.
stem, method = (stem_for('articleType'), winners['articleType'])
path = save_checkpoint(  # Bundle the fitted model with everything required to reproduce inference.
    checkpoints['articleType'], OUTPUT / f'{stem}_model.keras'
)
histories['articleType'][method].to_csv(RESULTS / f'{stem}_history.csv', index=False)
summary = dict(
    target='articleType',
    selected=method,
    cnn_selected=SELECTED_CONFIGS['articleType']['cnn']['method'],
    framework='tensorflow_keras',
    model_file=path.name,
    test_metrics=test_results['articleType']['metrics'],
    split_sizes={name: len(frame) for name, frame in frames['articleType'].items()},
    test_scope='internal_test_with_prior_development_exposure',
)
(RESULTS / f'{stem}_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print('Saved', path)

### 7.1. Load the Saved Model & Predict

A successful reload checks the saved architecture and metadata. This example is a functional check, not an independent quality estimate.


In [ ]:
# Reload an exported Keras classifier and recover the metadata stored inside it.
def resolve_classifier_path(path):
    path = Path(path)
    if path.exists():
        return path
    raise FileNotFoundError(
        f"Missing trained classifier: {path}. Run the classification notebook first."
    )

def load_checkpoint(path):
    path = resolve_classifier_path(path)
    if path.suffix == ".keras":
        model = keras.models.load_model(path, compile=False)
        return {**dict(model.get_layer("metadata").metadata), "model": model}
    raise ValueError(f"Unsupported classifier format: {path.suffix}")

In [ ]:
# Run a single-image smoke test through the same preprocessing path used by the application.
checkpoint = load_checkpoint(OUTPUT / f"{stem_for('articleType')}_model.keras")

# Apply the embedded preprocessing values rather than notebook globals.
with Image.open(frames['articleType']['selection'].image_path.iloc[0]) as image:
    resized = image.convert('RGB').resize(
        tuple(checkpoint['image_size']), Image.Resampling.BILINEAR
    )
    inputs = (
        np.asarray(resized, dtype=np.float32) / 255.0 - np.asarray(checkpoint['mean'])
    ) / np.asarray(checkpoint['std'])
probabilities = tf.nn.softmax(
    checkpoint['model'](inputs[None, ...], training=False), axis=-1
).numpy()[0]
indices = np.argsort(probabilities)[::-1][:3]
print(
    {
        'target': 'articleType',
        'label': checkpoint['labels'][int(indices[0])],
        'confidence': float(probabilities[indices[0]]),
        'top_k': [
            {'label': checkpoint['labels'][int(i)], 'confidence': float(probabilities[i])}
            for i in indices
        ],
    }
)

## 8. Conclusion

This task developed and evaluated an end-to-end image-classification pipeline for predicting article type from catalogue photographs. All three model families used the same audited images, frozen group-aware data split, RGB normalization and selection criterion. This makes the comparison a controlled test of how the model architecture affects performance. Accuracy measures the proportion of correct predictions, while supported-class macro-F1 gives equal importance to each class present in the evaluation partition and is therefore the primary measure for this highly imbalanced target.

The shallow MLP established a meaningful ANN baseline with **74.21% selection accuracy** and **0.4432 macro-F1**. The deeper MLP reached **73.24% accuracy** and **0.3783 macro-F1**, showing that additional fully connected depth did not compensate for flattening the image and losing explicit spatial relationships. Both dense models also required approximately 9.5 million parameters. The CNN achieved **87.00% selection accuracy** and **0.6954 macro-F1** with only **684,604 parameters**, roughly 93% fewer parameters than the MLPs. This demonstrates that parameter count alone does not determine performance; convolution provides a more suitable inductive structure for learning garment shape, texture and local visual patterns.

The per-class results reinforce this decision. The CNN performed strongly on high-support categories such as Tshirts, Shirts and Watches and also recognized several moderately supported classes reliably. It reduced the number of represented classes with zero recall to 13, compared with 28 for the shallow MLP and 38 for the deeper MLP. However, Bangle, Capris, Jackets and several extremely rare labels remained difficult. The training accuracy of 96.34% compared with 87.00% on the selection partition also indicates some overfitting. Early stopping, learning-rate reduction and restoration of the best macro-F1 epoch limit this effect, but they do not replace additional representative data.

On the internal-test partition, the selected CNN achieved **86.34% accuracy** and **0.6373 macro-F1**. The similar accuracy and lower macro-F1 confirm that the main visual patterns generalize, while rare-class reliability remains the principal limitation. Because this internal test has prior development exposure, these values describe performance within the supplied dataset rather than an independent estimate for all real-world fashion images. Repeated training seeds and a new external test set would be needed for a stronger generalization claim.

The selected CNN is exported as a self-contained Keras model with its label order, input size and normalization metadata. It supports article-type prediction in the web application, and its final hidden representation is reused by Task 4 for visual-search embeddings. The experiment therefore produces both a technically justified classifier and a deployable component. Future improvement should focus on collecting more examples for rare article types, reviewing ambiguous catalogue labels, and testing targeted augmentation or transfer learning under the same frozen evaluation protocol.
